# TaniAdaptif Live Inference Sidang

Notebook Kaggle ini hanya menjalankan inference Stable Diffusion 1.5 dengan LoRA final.
Notebook tidak memuat training loop, optimizer, dataset training, atau checkpoint training.

- Run ID: `20260820T143551Z`
- LoRA: `multisource_multicrop_lora.safetensors`
- SHA-256: `0bda8908d577513110c22cf5ec1b8a7e008d2b26dd90525732a5c4e4b1bdd43a`
- Private API: FastAPI port `5000`
- Public transport: Cloudflare Quick Tunnel

Sebelum **Run All**, aktifkan GPU T4, Internet, input dataset LoRA private, dan Kaggle User Secret `INFERENCE_API_TOKEN`.

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import platform
import re
import subprocess
import sys
import time
import urllib.error
import urllib.parse
import urllib.request
from pathlib import Path

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

KAGGLE_ROOT = Path("/kaggle")
INPUT_ROOT = KAGGLE_ROOT / "input"
WORK_ROOT = KAGGLE_ROOT / "working" / "taniadaptif_live_inference"
GENERATED_DIR = WORK_ROOT / "generated"
RUNTIME_ROOT = WORK_ROOT / "runtime"
HF_HOME = WORK_ROOT / "huggingface_cache"
FASTAPI_HOST = "127.0.0.1"
FASTAPI_PORT = 5000

BASE_MODEL = "stable-diffusion-v1-5/stable-diffusion-v1-5"
LORA_FILENAME = "multisource_multicrop_lora.safetensors"
LORA_SHA256 = "0bda8908d577513110c22cf5ec1b8a7e008d2b26dd90525732a5c4e4b1bdd43a"
LORA_IDENTIFIER = "multisource_multicrop_lora_sd15_20260820T143551Z"

if not WORK_ROOT.parent.is_dir() or not INPUT_ROOT.is_dir():
    raise SystemExit("STOP: notebook ini hanya boleh dijalankan pada runtime Kaggle.")
WORK_ROOT.mkdir(parents=True, exist_ok=True)
GENERATED_DIR.mkdir(parents=True, exist_ok=True)
HF_HOME.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(HF_HOME)

import torch

if not torch.cuda.is_available() or torch.cuda.device_count() < 1:
    raise SystemExit("STOP: CUDA tidak tersedia. Aktifkan Kaggle GPU T4 lalu Run All kembali.")

gpu = torch.cuda.get_device_properties(0)
gpu_vram_gib = round(int(gpu.total_memory) / (1024**3), 2)
if int(gpu.total_memory) < 12 * 1024**3:
    raise SystemExit(f"STOP: VRAM GPU tidak cukup: {gpu_vram_gib} GiB")

print("GPU detected:", gpu.name)
print("CUDA runtime:", torch.version.cuda)
print("PyTorch:", torch.__version__)
print("VRAM GiB:", gpu_vram_gib)
subprocess.run(["nvidia-smi"], check=False)

In [ ]:
DEPENDENCIES = [
    "fastapi==0.141.1",
    "starlette==1.3.1",
    "uvicorn[standard]==0.52.1",
    "pydantic==2.13.4",
    "pydantic-settings==2.14.2",
    "diffusers==0.35.1",
    "transformers==4.57.6",
    "accelerate==1.14.0",
    "peft==0.20.0",
    "safetensors==0.8.0",
    "huggingface-hub==0.36.2",
]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", *DEPENDENCIES],
    check=True,
)

# SD 1.5 fp16 tidak memakai TorchAO; hapus paket opsional Kaggle yang inkompatibel.
import importlib.util
subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "--quiet", "--yes", "torchao"],
    check=True,
)
for module_name in tuple(sys.modules):
    if module_name == "torchao" or module_name.startswith("torchao."):
        del sys.modules[module_name]
importlib.invalidate_caches()
if importlib.util.find_spec("torchao") is not None:
    raise SystemExit("STOP: torchao masih terdeteksi setelah uninstall.")
print("TORCHAO: ABSENT (not required)")

# Pillow dipasang paling akhir agar dependency lain tidak mengganti versinya.
PILLOW_VERSION = "11.3.0"
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--force-reinstall",
        "--no-cache-dir",
        "--no-deps",
        f"Pillow=={PILLOW_VERSION}",
    ],
    check=True,
)

# Kaggle dapat memuat PIL sebelum pip selesai; impor ulang dari instalasi final.
for module_name in tuple(sys.modules):
    if module_name == "PIL" or module_name.startswith("PIL."):
        del sys.modules[module_name]
importlib.invalidate_caches()
import PIL
from PIL import Image, ImageDraw

if PIL.__version__ != PILLOW_VERSION:
    raise SystemExit(
        f"STOP: Pillow version mismatch: {PIL.__version__} != {PILLOW_VERSION}"
    )
pillow_probe = Image.new("RGB", (2, 2), color="black")
ImageDraw.Draw(pillow_probe).point((0, 0), fill="white")
print(f"PILLOW VERSION: {PIL.__version__}")
print("PIL IMPORT: PASS")

CLOUDFLARED_PATH = WORK_ROOT / "cloudflared"
if not CLOUDFLARED_PATH.is_file():
    if platform.machine().lower() not in {"x86_64", "amd64"}:
        raise SystemExit(f"STOP: arsitektur cloudflared tidak didukung: {platform.machine()}")
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        CLOUDFLARED_PATH,
    )
    CLOUDFLARED_PATH.chmod(0o755)
subprocess.run([str(CLOUDFLARED_PATH), "--version"], check=True)
print("Runtime dependencies: PASS")

In [ ]:
from safetensors import safe_open


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


all_lora_matches = sorted(path for path in INPUT_ROOT.rglob(LORA_FILENAME) if path.is_file())
matching_lora = [path for path in all_lora_matches if sha256_file(path) == LORA_SHA256]
if len(matching_lora) != 1:
    observed = [{"path": str(path), "sha256": sha256_file(path)} for path in all_lora_matches]
    raise SystemExit(
        "STOP: harus ada tepat satu LoRA final dengan SHA yang benar. Observed: "
        + json.dumps(observed, indent=2)
    )

LORA_PATH = matching_lora[0]
if LORA_PATH.stat().st_size < 1024:
    raise SystemExit("STOP: LoRA final kosong atau terlalu kecil.")
with safe_open(LORA_PATH, framework="pt", device="cpu") as handle:
    tensor_keys = list(handle.keys())
    nonempty_tensors = sum(handle.get_tensor(key).numel() > 0 for key in tensor_keys)
    metadata_text = json.dumps(handle.metadata() or {}, sort_keys=True).casefold()
if len(tensor_keys) < 8 or nonempty_tensors < 8:
    raise SystemExit("STOP: LoRA final tidak memuat tensor adapter yang cukup.")
if any(marker in metadata_text for marker in ("placeholder", "dummy artifact", "synthetic artifact")):
    raise SystemExit("STOP: LoRA terdeteksi sebagai placeholder.")

print("LoRA path:", LORA_PATH)
print("LoRA bytes:", LORA_PATH.stat().st_size)
print("LoRA tensors:", len(tensor_keys))
print("LoRA SHA-256 PASS:", LORA_SHA256)

In [ ]:
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
SECRET_FALLBACK_PATH = (
    INPUT_ROOT
    / "taniadaptif-sidang-secret-private"
    / "inference_api_token.txt"
)
secret_source = "kaggle_user_secret"
try:
    INFERENCE_API_TOKEN = secrets.get_secret("INFERENCE_API_TOKEN").strip()
except Exception:
    secret_source = "private_credential_dataset"
    if not SECRET_FALLBACK_PATH.is_file():
        raise SystemExit(
            "STOP: Kaggle User Secret gagal dan private credential dataset tidak tersedia."
        )
    if SECRET_FALLBACK_PATH.stat().st_size != 64:
        raise SystemExit("STOP: ukuran credential private dataset bukan 64 byte.")
    INFERENCE_API_TOKEN = SECRET_FALLBACK_PATH.read_text(encoding="utf-8").strip()

if len(INFERENCE_API_TOKEN) != 64 or INFERENCE_API_TOKEN.casefold().startswith("replace-with"):
    raise SystemExit("STOP: INFERENCE_API_TOKEN harus non-placeholder dan tepat 64 karakter.")

try:
    HF_TOKEN = secrets.get_secret("HF_TOKEN").strip()
except Exception:
    HF_TOKEN = ""
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN

print("Required secret loaded: INFERENCE_API_TOKEN")
print("SECRET STATUS=PASS")
print("TOKEN LENGTH=64")
print("Secret source:", secret_source)
print("Optional secret HF_TOKEN:", "configured" if HF_TOKEN else "not required/not configured")

In [ ]:
import base64
import io
import shutil
import zipfile

RUNTIME_ARCHIVE_SHA256 = '57983ccba7860f4d8dafc2b27170270fd8c64349ad541a1a68ab64add3003dfa'
RUNTIME_ARCHIVE_B64 = '...'
RUNTIME_SOURCE_MANIFEST = '{"inference_api/app/__init__.py":"abb5fab205e56925415c22f6377905d558305c659b5db00af87f6961c6eb2618","inference_api/app/api/__init__.py":"d93b5f927d5548a8ac36fcb08ff9ea4d61844066f86244c9564b8b01c01428b4","inference_api/app/api/routes.py":"f856ab51080e273cb438c5e767209667807df1982a8549a29a07ba00418539f0","inference_api/app/core/__init__.py":"cbfe6b89cb869af3e65ac9a63c362a967caabea9dd46246b5cf8830e15af07be","inference_api/app/core/config.py":"d8e6afcea6714e44fb92fd9bd0a3ee229f08c0b37385d42ca9e282803cbfc0b5","inference_api/app/core/errors.py":"6dd9efe972f0c7f5313fb893aaa91570f3b3b2eb7d388a4e3b9b969940795c7f","inference_api/app/main.py":"a515f99b240d34b6465e46a3d5cde4cc15513b03e11b85755e5895f301472998","inference_api/app/models/__init__.py":"0898b02a1665e1da76b79fb770f79de3ab4281dcd566651ba4b9f3e8d52f3920","inference_api/app/models/schemas.py":"df496a843636f47580f314d0f5860dd7352d290db3cdc0647a4ad3aa1ba87dc8","inference_api/app/providers/__init__.py":"ea78abe5924b9ee4c67f896ce11db4bf0050cbd5b490395e5eef7bd3182e931e","inference_api/app/providers/base.py":"187920a833aec96b05d360c80949e2f573b4dac920c08bb945d85f08ed3eb84f","inference_api/app/providers/diffusers_provider.py":"d81561f2093ab3f812e6615b0da800fd15c06a873ddfba1042edcbbe4f5e4fca","inference_api/app/providers/lora_validation.py":"ea16e804b4c0f207add5d7dcf1e247cdfe3dc1af030c725540ab25ddf26c0367","inference_api/app/providers/mock.py":"5f10093654452f7961c55ab3872a9de6384fb4cf2c11fc95e694c339436fe071","inference_api/app/services/__init__.py":"ea99f2f0f3b51e94912e9161c705257c74e5a8faf58f44e05b5517515d9e2373","inference_api/app/services/generation_service.py":"810dcd53c5f8404b7f43ed9792bc255aa626282ed94ec2092aecd0cbfc147587","inference_api/app/services/prompt_builder.py":"d7a7089ac570fa2a0c808ea00949bbb774644a9af57c6c422b395dde5c0cd7ed"}'
RUNTIME_SOURCE_MANIFEST = json.loads(RUNTIME_SOURCE_MANIFEST)

archive_payload = base64.b64decode(RUNTIME_ARCHIVE_B64, validate=True)
if hashlib.sha256(archive_payload).hexdigest() != RUNTIME_ARCHIVE_SHA256:
    raise SystemExit("STOP: embedded inference_api archive hash mismatch.")
if RUNTIME_ROOT.exists():
    shutil.rmtree(RUNTIME_ROOT)
RUNTIME_ROOT.mkdir(parents=True)
with zipfile.ZipFile(io.BytesIO(archive_payload)) as archive:
    unsafe = [name for name in archive.namelist() if Path(name).is_absolute() or ".." in Path(name).parts]
    if unsafe:
        raise SystemExit(f"STOP: unsafe embedded runtime paths: {unsafe}")
    archive.extractall(RUNTIME_ROOT)

for relative, expected in RUNTIME_SOURCE_MANIFEST.items():
    extracted = RUNTIME_ROOT / relative
    if not extracted.is_file() or sha256_file(extracted) != expected:
        raise SystemExit(f"STOP: embedded runtime source mismatch: {relative}")
sys.path.insert(0, str(RUNTIME_ROOT))
print("Embedded inference_api snapshot: PASS")
print("Archive SHA-256:", RUNTIME_ARCHIVE_SHA256)
print("Source files:", len(RUNTIME_SOURCE_MANIFEST))

In [ ]:
os.environ.update(
    {
        "INFERENCE_PROVIDER": "diffusers",
        "BASE_MODEL": BASE_MODEL,
        "LORA_PATH": str(LORA_PATH),
        "LORA_SHA256": LORA_SHA256,
        "LORA_IDENTIFIER": LORA_IDENTIFIER,
        "REQUIRE_GPU": "true",
        "INFERENCE_REQUIRE_TOKEN": "true",
        "INFERENCE_API_TOKEN": INFERENCE_API_TOKEN,
        "GENERATED_DIR": str(GENERATED_DIR),
        "CORS_ORIGINS": "",
        "GENERATION_TIMEOUT_SECONDS": "85",
        "GENERATION_MAX_RETRIES": "1",
        "DEFAULT_SEED": "42",
    }
)

from inference_api.app.core.config import Settings
from inference_api.app.main import create_app
from inference_api.app.providers.lora_validation import validate_lora_artifact
from inference_api.app.services.generation_service import GenerationService

SETTINGS = Settings(_env_file=None)
LORA_INFO = validate_lora_artifact(SETTINGS.lora_path, SETTINGS.lora_sha256)
INFERENCE_SERVICE = GenerationService(SETTINGS)
MODEL_HEALTH = INFERENCE_SERVICE.health()
if not (
    MODEL_HEALTH.get("provider") == "diffusers"
    and MODEL_HEALTH.get("device") == "cuda"
    and MODEL_HEALTH.get("model_loaded") is True
    and MODEL_HEALTH.get("lora_sha256") == LORA_SHA256
):
    raise SystemExit("STOP: model/LoRA gagal dimuat: " + json.dumps(MODEL_HEALTH, indent=2))

FASTAPI_APP = create_app(SETTINGS, INFERENCE_SERVICE)
print("Model loaded once: PASS")
print(json.dumps(MODEL_HEALTH, indent=2))

In [ ]:
import threading
import uvicorn


def request_json(url: str, *, token: str = "", method: str = "GET", body=None, timeout=30):
    headers = {"Accept": "application/json", "User-Agent": "TaniAdaptif-Sidang/1.0"}
    if token:
        headers["X-Inference-Token"] = token
    payload = None
    if body is not None:
        headers["Content-Type"] = "application/json"
        payload = json.dumps(body).encode("utf-8")
    request = urllib.request.Request(url, data=payload, headers=headers, method=method)
    with urllib.request.urlopen(request, timeout=timeout) as response:
        return json.loads(response.read().decode("utf-8"))


UVICORN_CONFIG = uvicorn.Config(
    FASTAPI_APP,
    host=FASTAPI_HOST,
    port=FASTAPI_PORT,
    log_level="info",
    access_log=False,
    proxy_headers=True,
    forwarded_allow_ips="127.0.0.1",
)
UVICORN_SERVER = uvicorn.Server(UVICORN_CONFIG)
UVICORN_THREAD = threading.Thread(target=UVICORN_SERVER.run, daemon=True)
UVICORN_THREAD.start()

LOCAL_URL = f"http://{FASTAPI_HOST}:{FASTAPI_PORT}"
deadline = time.time() + 30
while time.time() < deadline:
    try:
        local_health = request_json(f"{LOCAL_URL}/health", timeout=3)
        if local_health.get("status") == "ok":
            break
    except Exception:
        time.sleep(1)
else:
    raise SystemExit("STOP: FastAPI gagal aktif pada port 5000.")

local_ready = request_json(f"{LOCAL_URL}/ready", token=INFERENCE_API_TOKEN, timeout=10)
if local_ready.get("status") != "ready":
    raise SystemExit("STOP: FastAPI aktif tetapi provider tidak ready.")
print("FastAPI local health: PASS")
print("FastAPI port:", FASTAPI_PORT)

In [ ]:
import queue

if "CLOUDFLARED_PROCESS" in globals() and CLOUDFLARED_PROCESS.poll() is None:
    CLOUDFLARED_PROCESS.terminate()
    CLOUDFLARED_PROCESS.wait(timeout=10)

cloudflared_home = WORK_ROOT / "cloudflared_home"
cloudflared_home.mkdir(parents=True, exist_ok=True)
cloudflared_env = {**os.environ, "HOME": str(cloudflared_home)}
CLOUDFLARED_PROCESS = subprocess.Popen(
    [
        str(CLOUDFLARED_PATH),
        "tunnel",
        "--url",
        LOCAL_URL,
        "--no-autoupdate",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=cloudflared_env,
)

tunnel_lines = queue.Queue()


def collect_tunnel_output():
    assert CLOUDFLARED_PROCESS.stdout is not None
    for line in CLOUDFLARED_PROCESS.stdout:
        tunnel_lines.put(line.rstrip())


threading.Thread(target=collect_tunnel_output, daemon=True).start()
tunnel_pattern = re.compile(r"https://[a-z0-9-]+\.trycloudflare\.com")
KAGGLE_GPU_URL = ""
deadline = time.time() + 90
while time.time() < deadline and CLOUDFLARED_PROCESS.poll() is None:
    try:
        line = tunnel_lines.get(timeout=1)
    except queue.Empty:
        continue
    match = tunnel_pattern.search(line)
    if match:
        KAGGLE_GPU_URL = match.group(0)
        break
if not KAGGLE_GPU_URL:
    raise SystemExit("STOP: Cloudflare Quick Tunnel gagal menghasilkan URL HTTPS.")

deadline = time.time() + 90
while time.time() < deadline:
    try:
        public_health = request_json(f"{KAGGLE_GPU_URL}/health", timeout=10)
        if public_health.get("status") == "ok":
            break
    except Exception:
        pass
    time.sleep(2)
else:
    raise SystemExit("STOP: URL Quick Tunnel tersedia tetapi health publik gagal.")

public_ready = {}
last_ready_error = "provider belum ready"
deadline = time.time() + 90
while time.time() < deadline:
    try:
        public_ready = request_json(
            f"{KAGGLE_GPU_URL}/ready",
            token=INFERENCE_API_TOKEN,
            timeout=20,
        )
        if public_ready.get("status") == "ready":
            break
        last_ready_error = f"status={public_ready.get('status', 'missing')}"
    except urllib.error.HTTPError as exc:
        if exc.code in {401, 403}:
            raise SystemExit(f"STOP: autentikasi readiness publik ditolak (HTTP {exc.code}).") from exc
        last_ready_error = f"HTTPError {exc.code}"
    except (urllib.error.URLError, TimeoutError, OSError) as exc:
        last_ready_error = f"{type(exc).__name__}: {getattr(exc, 'reason', exc)}"
    time.sleep(2)
else:
    raise SystemExit(
        "STOP: readiness publik bukan ready setelah retry. "
        f"Error terakhir: {last_ready_error}"
    )
print(f"KAGGLE_GPU_URL={KAGGLE_GPU_URL}")
print("Cloudflare tunnel health: PASS")

In [ ]:
SANITY_REQUEST_ID = "kaggle-sidang-sanity-42"
sanity_response = request_json(
    f"{KAGGLE_GPU_URL}/v1/generate",
    token=INFERENCE_API_TOKEN,
    method="POST",
    body={
        "crop": "padi",
        "land_type": "sawah",
        "water_availability": "cukup",
        "previous_crop": "jagung",
        "weather_summary": "Cerah berawan; suhu 28 C; kelembaban 78 persen",
        "seed": 42,
        "request_id": SANITY_REQUEST_ID,
        "steps": 15,
        "guidance_scale": 7.5,
        "width": 512,
        "height": 512,
    },
    timeout=100,
)
metadata = sanity_response.get("metadata") or {}
sanity_checks = {
    "success": sanity_response.get("success") is True,
    "request_id": sanity_response.get("request_id") == SANITY_REQUEST_ID,
    "model": sanity_response.get("model") == BASE_MODEL,
    "provider": metadata.get("provider") == "diffusers",
    "device": metadata.get("device") == "cuda",
    "not_mock": metadata.get("is_mock") is False,
    "lora_sha256": metadata.get("lora_sha256") == LORA_SHA256,
}
if not all(sanity_checks.values()):
    raise SystemExit("STOP: sanity metadata gagal: " + json.dumps(sanity_checks, indent=2))

image_path_from_api = urllib.parse.urlsplit(str(sanity_response.get("image_url") or "")).path
if not image_path_from_api.startswith("/static/generated/"):
    raise SystemExit("STOP: sanity image_url tidak menggunakan static/generated.")
public_image_url = urllib.parse.urljoin(KAGGLE_GPU_URL, image_path_from_api)
image_request = urllib.request.Request(
    public_image_url,
    headers={
        "Accept": "image/png",
        "User-Agent": "TaniAdaptif-Sidang/1.0",
        "X-Inference-Token": INFERENCE_API_TOKEN,
    },
)
with urllib.request.urlopen(image_request, timeout=30) as response:
    image_content_type = response.headers.get_content_type()
    image_bytes = response.read()
if image_content_type != "image/png" or image_bytes[:8] != b"\x89PNG\r\n\x1a\n":
    raise SystemExit("STOP: sanity artifact bukan PNG valid.")
if len(image_bytes) < 24 or image_bytes[12:16] != b"IHDR":
    raise SystemExit("STOP: sanity PNG tidak memiliki IHDR valid.")
width = int.from_bytes(image_bytes[16:20], "big")
height = int.from_bytes(image_bytes[20:24], "big")
if (width, height) != (512, 512):
    raise SystemExit(f"STOP: dimensi sanity PNG salah: {width}x{height}")

SANITY_PATH = WORK_ROOT / "sanity_inference.png"
SANITY_PATH.write_bytes(image_bytes)
SANITY_SHA256 = hashlib.sha256(image_bytes).hexdigest()
print("Sanity inference PNG: PASS")
print("Sanity path:", SANITY_PATH)
print("Sanity bytes:", len(image_bytes))
print("Sanity SHA-256:", SANITY_SHA256)

In [ ]:
model_info = request_json(
    f"{KAGGLE_GPU_URL}/v1/model-info",
    token=INFERENCE_API_TOKEN,
    timeout=20,
)
if not (
    model_info.get("base_model") == BASE_MODEL
    and model_info.get("lora_identifier") == LORA_IDENTIFIER
    and model_info.get("provider") == "diffusers"
):
    raise SystemExit("STOP: model-info tidak sesuai model final.")

STATE_PATH = WORK_ROOT / "kaggle_live_inference_state.json"
state = {
    "status": "success",
    "training_performed": False,
    "gpu": gpu.name,
    "fastapi_port": FASTAPI_PORT,
    "kaggle_gpu_url": KAGGLE_GPU_URL,
    "health": public_health,
    "ready": public_ready,
    "base_model": BASE_MODEL,
    "lora_identifier": LORA_IDENTIFIER,
    "lora_sha256": LORA_SHA256,
    "sanity_inference": {
        "status": "success",
        "path": str(SANITY_PATH),
        "sha256": SANITY_SHA256,
        "bytes": len(image_bytes),
        "width": width,
        "height": height,
    },
    "secret_names": ["INFERENCE_API_TOKEN", "HF_TOKEN (optional)"],
    "secret_values_recorded": False,
}
STATE_PATH.write_text(json.dumps(state, indent=2) + "\n", encoding="utf-8")

print("GPU=" + gpu.name)
print("FASTAPI_PORT=5000")
print("SECRET_NAMES=INFERENCE_API_TOKEN,HF_TOKEN(optional)")
print("KAGGLE_GPU_URL=" + KAGGLE_GPU_URL)
print("HEALTH=PASS")
print("SANITY_INFERENCE=PASS")
print("STATUS=READY_FOR_SIDANG")

In [ ]:
KEEP_ALIVE_MINUTES = 330
print(f"Service live selama maksimal {KEEP_ALIVE_MINUTES} menit. Biarkan cell ini berjalan saat sidang.")
for elapsed_minutes in range(KEEP_ALIVE_MINUTES):
    if not UVICORN_THREAD.is_alive():
        raise RuntimeError("FastAPI thread berhenti.")
    if CLOUDFLARED_PROCESS.poll() is not None:
        raise RuntimeError("Cloudflare tunnel berhenti.")
    if elapsed_minutes % 15 == 0:
        check = request_json(f"{LOCAL_URL}/health", timeout=5)
        if check.get("status") != "ok":
            raise RuntimeError("FastAPI health tidak lagi ok.")
        print(f"KEEP_ALIVE minute={elapsed_minutes} KAGGLE_GPU_URL={KAGGLE_GPU_URL}")
    time.sleep(60)
print("Keep-alive selesai. Run ulang notebook untuk membuat tunnel baru.")